In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
import os, gc
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl

In [2]:
IS_SAMPLE      = False
NEGATIVE_RATIO = 4
CHUNK_SIZE     = 500_000

PROCESSED_DATA_DIR = '/kaggle/input/datasets/b22dckh072/file05'
TRAIN_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(PROCESSED_DATA_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(PROCESSED_DATA_DIR, 'candidates_phase2.parquet')
TEST_PATH = os.path.join(PROCESSED_DATA_DIR, 'test_interactions.parquet')
FEAT_OUT   = os.path.join('/kaggle/working/features.parquet')

In [3]:
print("Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)
lf_cands = pl.scan_parquet(CAND_PATH)

print("Bước 2: Tính toán đặc trưng thống kê (User Stats & Item Stats)...")
user_stats = lf_train.group_by('mapped_user_id').agg([
    pl.len().alias('user_total_actions'),
    pl.col('rating').mean().alias('user_avg_rating_given')
])

item_stats = lf_train.group_by('mapped_item_id').agg([
    pl.len().alias('item_total_sales'),
    pl.col('rating').mean().alias('item_actual_avg_rating')
])

print("Bước 3: Đánh nhãn trực tiếp trên tập Ứng viên (Chống Data Leakage)...")

lf_test = pl.scan_parquet(TEST_PATH)

# 1. Chia tập Test: Lấy các người dùng chẵn để làm Validation
lf_val = lf_test.filter((pl.col('mapped_user_id') % 2) == 0)
valid_users = lf_val.select('mapped_user_id').unique()

# 2. CHỈ LẤY Candidates của những user này (Bắt buộc phải xuất phát từ Candidates)
lf_cands_val = lf_cands.join(valid_users, on='mapped_user_id', how='inner')

# 3. Đánh nhãn: Nếu Candidate trúng với đồ mua trong tương lai -> Nhãn 1, ngược lại -> Nhãn 0
lf_labels = (
    lf_cands_val.select(['mapped_user_id', 'mapped_item_id'])
    .join(
        lf_val.select(['mapped_user_id', 'mapped_item_id']).with_columns(pl.lit(1).alias('label').cast(pl.Int8)),
        on=['mapped_user_id', 'mapped_item_id'],
        how='left'
    )
    .with_columns(pl.col('label').fill_null(0).cast(pl.Int8)) # Điền Nhãn 0 cho những món đoán sai
)

# 4. Loại bỏ những món Candidates trùng với đồ đã mua trong quá khứ
lf_labels_clean = lf_labels.join(
    lf_train.select(['mapped_user_id', 'mapped_item_id']),
    on=['mapped_user_id', 'mapped_item_id'],
    how='anti'
)

# Thu thập toàn bộ nhãn
df_all_labels = lf_labels_clean.collect(engine="streaming")

# 5. Cân bằng dữ liệu theo NEGATIVE_RATIO
df_positives = df_all_labels.filter(pl.col('label') == 1)
df_negatives = df_all_labels.filter(pl.col('label') == 0)

df_sampled_negatives = (
    df_negatives.group_by('mapped_user_id', maintain_order=True)
    .head(NEGATIVE_RATIO)
)

# Gộp lại thành bảng df_labels cuối cùng
df_labels = pl.concat([df_positives, df_sampled_negatives])
print(f"Tổng số mẫu dương tính (Nhãn 1 - Hit): {df_positives.height:,}")
print(f"Tổng số mẫu huấn luyện Ranker (Dòng): {df_labels.height:,}")


print("Bước 4: Nối Đặc trưng theo từng khối (Tối ưu RAM tuyệt đối)...")
import os
import shutil

# 1. TÍNH TOÁN VÀ NẠP CÁC BẢNG PHỤ VÀO RAM TRƯỚC
# Việc này đảm bảo Polars không phải tính toán lại hàng chục triệu dòng ở mỗi vòng lặp
print("-> Đang nạp các bảng thống kê và Meta vào bộ nhớ...")
df_meta_mem = lf_meta.collect()
df_user_stats_mem = user_stats.collect()
df_item_stats_mem = item_stats.collect()

lf_ranks = lf_cands.select(['mapped_user_id', 'mapped_item_id', 'sasrec_rank', 'lightgcn_rank'])

CHUNK_SIZE = 2_000_000
total_rows = df_labels.height
TEMP_DIR = "feat_chunks_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

for start_idx in range(0, total_rows, CHUNK_SIZE):
    end_idx = min(start_idx + CHUNK_SIZE, total_rows)
    print(f"-> Đang xử lý khối {start_idx:,} đến {end_idx:,}...")
    
    # Cắt khối nhỏ 2 triệu dòng
    chunk_df = df_labels.slice(start_idx, CHUNK_SIZE)
    
    # 2. LỌC BẢNG CANDIDATES TRƯỚC KHI JOIN
    # Chỉ lấy rank của những user có mặt trong khối hiện tại để tránh phình to RAM
    unique_users = chunk_df['mapped_user_id'].unique().to_list()
    df_ranks_chunk = lf_ranks.filter(pl.col('mapped_user_id').is_in(unique_users)).collect()
    
    # 3. THỰC HIỆN JOIN TRÊN CÁC BẢNG ĐÃ CÓ SẴN TRONG RAM
    chunk_processed = (
        chunk_df.lazy()
        .join(df_ranks_chunk.lazy(), on=['mapped_user_id', 'mapped_item_id'], how='left')
        .join(df_meta_mem.lazy(), on='mapped_item_id', how='left')
        .join(df_user_stats_mem.lazy(), on='mapped_user_id', how='left')
        .join(df_item_stats_mem.lazy(), on='mapped_item_id', how='left')
        .with_columns([
            (1.0 / (pl.col('sasrec_rank') + 1.0)).fill_null(0.0).alias('sasrec_score'),
            (1.0 / (pl.col('lightgcn_rank') + 1.0)).fill_null(0.0).alias('lightgcn_score'),
            
            pl.col('price').fill_null(0.0),
            pl.col('average_rating').fill_null(0.0),
            pl.col('rating_number').fill_null(0),
            pl.col('user_total_actions').fill_null(0),
            pl.col('item_total_sales').fill_null(0),
            pl.col('user_avg_rating_given').fill_null(3.0)
        ])
        .drop(['sasrec_rank', 'lightgcn_rank'])
        .collect()
    )
    
    # Ghi khối nhỏ ra đĩa
    chunk_processed.write_parquet(f"{TEMP_DIR}/chunk_{start_idx}.parquet")
    
    # Dọn dẹp RAM ngay lập tức
    del chunk_df, df_ranks_chunk, chunk_processed, unique_users
    gc.collect()

print("-> Đang hợp nhất các khối thành file Features hoàn chỉnh...")
pl.scan_parquet(f"{TEMP_DIR}/*.parquet").sink_parquet(FEAT_OUT)

# Dọn dẹp toàn bộ rác trung gian
shutil.rmtree(TEMP_DIR)
del df_labels, lf_train, lf_meta, lf_cands, user_stats, item_stats
del df_meta_mem, df_user_stats_mem, df_item_stats_mem
gc.collect()

print(f"Hoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: {FEAT_OUT}")

Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...
Bước 2: Tính toán đặc trưng thống kê (User Stats & Item Stats)...
Bước 3: Đánh nhãn trực tiếp trên tập Ứng viên (Chống Data Leakage)...
Tổng số mẫu dương tính (Nhãn 1 - Hit): 54,975
Tổng số mẫu huấn luyện Ranker (Dòng): 2,409,779
Bước 4: Nối Đặc trưng theo từng khối (Tối ưu RAM tuyệt đối)...
-> Đang nạp các bảng thống kê và Meta vào bộ nhớ...
-> Đang xử lý khối 0 đến 2,000,000...
-> Đang xử lý khối 2,000,000 đến 2,409,779...
-> Đang hợp nhất các khối thành file Features hoàn chỉnh...
Hoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: /kaggle/working/features.parquet


In [4]:
print(lf_ranks.head().collect())

shape: (5, 4)
┌────────────────┬────────────────┬─────────────┬───────────────┐
│ mapped_user_id ┆ mapped_item_id ┆ sasrec_rank ┆ lightgcn_rank │
│ ---            ┆ ---            ┆ ---         ┆ ---           │
│ i64            ┆ i32            ┆ i8          ┆ i8            │
╞════════════════╪════════════════╪═════════════╪═══════════════╡
│ 0              ┆ 0              ┆ null        ┆ 1             │
│ 0              ┆ 5              ┆ null        ┆ 2             │
│ 0              ┆ 7              ┆ null        ┆ 3             │
│ 0              ┆ 8              ┆ null        ┆ 4             │
│ 0              ┆ 11             ┆ null        ┆ 5             │
└────────────────┴────────────────┴─────────────┴───────────────┘
